In [ ]:
!pip install catboost

In [ ]:
import os
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score,log_loss

import warnings
warnings.filterwarnings('ignore')
import numpy as np
from scipy.sparse import csr_matrix,hstack
from sklearn.pipeline import Pipeline

In [296]:
if os.path.exists('Dataset/train.csv'):
    DATA_DIR = 'Dataset'
elif os.path.exists('/kaggle/input/competitions/mlp-jan-2026-kaggle-assignment-2/train.csv'):
    DATA_DIR = '/kaggle/input/competitions/mlp-jan-2026-kaggle-assignment-2'
else:
    raise FileNotFoundError('Could not find train.csv in Dataset/ or the Kaggle competition input directory.')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
train = train.drop(columns=['ID'])
test = pd.read_csv(f'{DATA_DIR}/test.csv')
test = test.drop(columns=['ID'])

In [297]:
train

,mushroom_id,cap-shape,cap-surface,cap-color,bruises,number_of_bruises,odor,gill-attachment,gill-spacing,gill-size,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,class
0,0,convex,smooth,brown,bruises,7,pungent,gills free from stalk,close,narrow,...,white,white,partial,white,1.0,pendant,black,scattered,urban,p
1,1,convex,smooth,yellow,bruises,20,almond,gills free from stalk,close,broad,...,white,white,partial,white,1.0,pendant,brown,numerous,grasses,e
2,3,convex,scaly,white,bruises,11,pungent,gills free from stalk,close,narrow,...,white,white,partial,white,1.0,pendant,black,scattered,urban,p
3,4,convex,smooth,gray,no,0,NaN,gills free from stalk,crowded,broad,...,white,white,partial,white,1.0,evanescent,brown,abundant,grasses,e
4,5,convex,scaly,yellow,bruises,8,almond,gills free from stalk,close,broad,...,white,white,partial,white,1.0,pendant,black,numerous,grasses,e
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,8111,knobbed,smooth,white,no,0,NaN,gills free from stalk,crowded,broad,...,white,white,partial,white,2.0,pendant,white,numerous,grasses,e
6996,8113,knobbed,scaly,red,no,0,fishy,gills free from stalk,close,narrow,...,pink,pink,partial,white,1.0,evanescent,white,several,woods,p
6997,8114,flat,scaly,cinnamon,no,0,musty,gills attached to stalk,close,broad,...,cinnamon,cinnamon,partial,white,NaN,NaN,white,clustered,woods,p
6998,8117,knobbed,smooth,red,no,0,fishy,gills free from stalk,close,narrow,...,pink,white,partial,white,1.0,evanescent,white,several,woods,p


In [298]:
X=train.drop(columns=['class'])
y=train['class']

train_cols = X.columns
test = test[train_cols]

In [299]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)
y = pd.Series(y)

In [300]:
def feat(df):
  df=df.copy()
  df['stalk-root']=df['stalk-root'].fillna('missing')
  df['odor']=df['odor'].fillna('missing')
  df['ring-number'] = df['ring-number'].fillna(df['ring-number'].mode()[0])
  df['ring-type'] = df['ring-type'].fillna(df['ring-type'].mode()[0])
  return df

In [301]:
X=feat(X)
test=feat(test)

In [302]:
cat_col=X.select_dtypes('object').columns

In [303]:
ohe=OneHotEncoder(handle_unknown='ignore',drop='first')

In [304]:
lgb_pipeline = Pipeline([
    ('model', LGBMClassifier(random_state=42))
])

xgb_pipeline = Pipeline([
    ('model', XGBClassifier(random_state=42))
])

cat_pipeline = Pipeline([
    ('model', CatBoostClassifier(random_state=42))
])

In [305]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_scores = []

for train_idx, val_idx in skf.split(X, y):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    X_train_ohe=ohe.fit_transform(X_train[cat_col])
    X_val_ohe=ohe.transform(X_val[cat_col])

    X_train_num=X_train.drop(columns=cat_col)
    X_val_num=X_val.drop(columns=cat_col)

    X_train_final=hstack((X_train_ohe,csr_matrix(X_train_num.values)))
    X_val_final=hstack((X_val_ohe,csr_matrix(X_val_num.values)))

    lgb_pipeline.fit(X_train_final, y_train)
    xgb_pipeline.fit(X_train_final, y_train)
    cat_pipeline.fit(X_train_final, y_train)

    lgb_pred = lgb_pipeline.predict(X_val_final)
    xgb_pred = xgb_pipeline.predict(X_val_final)
    cat_pred = cat_pipeline.predict(X_val_final)

    lgb_acc = accuracy_score(y_val, lgb_pred)
    xgb_acc = accuracy_score(y_val, xgb_pred)
    cat_acc = accuracy_score(y_val, cat_pred)
    print(lgb_acc, xgb_acc, cat_acc)

    total = lgb_acc + xgb_acc + cat_acc
    w_lgb = lgb_acc / total
    w_xgb = xgb_acc / total
    w_cat = cat_acc / total

    lgb_proba = lgb_pipeline.predict_proba(X_val_final)
    xgb_proba = xgb_pipeline.predict_proba(X_val_final)
    cat_proba = cat_pipeline.predict_proba(X_val_final)

    avg_proba = (
        w_lgb * lgb_proba +
        w_xgb * xgb_proba +
        w_cat * cat_proba
    )

    lgb_loss = log_loss(y_val, lgb_proba)
    xgb_loss = log_loss(y_val, xgb_proba)
    cat_loss = log_loss(y_val, cat_proba)

    lgb_w = 1 / lgb_loss
    xgb_w = 1 / xgb_loss
    cat_w = 1 / cat_loss

    total = lgb_w + xgb_w + cat_w

    w_lgb = lgb_w / total
    w_xgb = xgb_w / total
    w_cat = cat_w / total

    final_pred = np.argmax(avg_proba, axis=1)

    acc = accuracy_score(y_val, final_pred)
    ensemble_scores.append(acc)

    print(f"Weights -> LGB: {w_lgb:.3f}, XGB: {w_xgb:.3f}, CAT: {w_cat:.3f}")

print("Final Ensemble Accuracy:", np.mean(ensemble_scores))

Streaming output truncated to the last 5000 lines.
385:	learn: 0.0005617	total: 1.88s	remaining: 2.99s
386:	learn: 0.0005617	total: 1.88s	remaining: 2.98s
387:	learn: 0.0005617	total: 1.88s	remaining: 2.97s
388:	learn: 0.0005617	total: 1.89s	remaining: 2.96s
389:	learn: 0.0005617	total: 1.89s	remaining: 2.95s
390:	learn: 0.0005617	total: 1.89s	remaining: 2.95s
391:	learn: 0.0005573	total: 1.89s	remaining: 2.94s
392:	learn: 0.0005573	total: 1.9s	remaining: 2.93s
393:	learn: 0.0005573	total: 1.9s	remaining: 2.92s
394:	learn: 0.0005573	total: 1.9s	remaining: 2.91s
395:	learn: 0.0005573	total: 1.91s	remaining: 2.91s
396:	learn: 0.0005573	total: 1.91s	remaining: 2.9s
397:	learn: 0.0005573	total: 1.91s	remaining: 2.89s
398:	learn: 0.0005499	total: 1.91s	remaining: 2.88s
399:	learn: 0.0005499	total: 1.92s	remaining: 2.88s
400:	learn: 0.0005499	total: 1.92s	remaining: 2.87s
401:	learn: 0.0005499	total: 1.92s	remaining: 2.86s
402:	learn: 0.0005499	total: 1.93s	remaining: 2.85s
403:	learn: 0.000

In [307]:
ohe.fit(X[cat_col])
test_ohe = ohe.transform(test[cat_col])
test_num = test.drop(columns=cat_col)
test_final = hstack((test_ohe, csr_matrix(test_num.values)))

lgb_proba = lgb_pipeline.predict_proba(test_final)
xgb_proba = xgb_pipeline.predict_proba(test_final)
cat_proba = cat_pipeline.predict_proba(test_final)

avg_proba = (
    w_lgb * lgb_proba +
    w_xgb * xgb_proba +
    w_cat * cat_proba
)

final_pred = np.argmax(avg_proba, axis=1)

final_pred_labels = le.inverse_transform(final_pred)

In [308]:
sample_path = f'{DATA_DIR}/sample_submission.csv'
if not os.path.exists(sample_path):
    sample_path = f'{DATA_DIR}/sample.csv'

sample = pd.read_csv(sample_path)

if len(final_pred_labels) != len(sample):
    raise ValueError(
        f'Prediction length mismatch: got {len(final_pred_labels)} predictions for {len(sample)} submission rows. '
        'Regenerate predictions using the Kaggle test set loaded from DATA_DIR.'
    )

sample['class'] = final_pred_labels
sample.to_csv('submission.csv', index=False)
sample.head()